In [1]:
from numpy.lib.stride_tricks import sliding_window_view
import numpy as np


def vec_pearson_corrcoef(X: np.array, Y: np.array, k):
    Sx  = X.sum(-1) 
    Sy  = Y.sum(-1)
    Sxx = (X * X).sum(-1)
    Syy = (Y * Y).sum(-1)
    Sxy = (X * Y).sum(-1)

    num = k*Sxy - Sx*Sy
    den = np.sqrt((k*Sxx - Sx*Sx) * (k*Syy - Sy*Sy))
    
    r = np.zeros_like(num, dtype=np.float32)
    np.divide(num, den, out=r, where=den!=0)
    return r



def sphere_mask(r):
    z,y,x = np.mgrid[-r:r+1, -r:r+1, -r:r+1]
    return (x*x + y*y + z*z) <= r*r   # Assumes isotropic


def rolling_pearson_corr(A, B, radius):
    W = (2*radius+1,)*3  # window shape 
    sph = sphere_mask(radius)  # sphere mask within window              
    k = int(sph.sum())
    

    A_wind = sliding_window_view(A, W)[..., sph]
    B_wind = sliding_window_view(B, W)[..., sph]

    # Reshape sliding window views so that windows are 1D - more efficient 
    window_size = np.prod(W)
    A_flat_wind = A_wind.reshape(-1, k)
    B_flat_wind = B_wind.reshape(-1, k)
    
    rho = vec_pearson_corrcoef(A_flat_wind, B_flat_wind, window_size)
    out_shape = tuple(d - 2*radius for d in A.shape)
    rho_valid = rho.reshape(out_shape)

    # Embed back into full-sized volume, padding AFTER the correlation
    out = np.full(A.shape, 0, dtype=np.float32)
    out[radius:-radius, radius:-radius, radius:-radius] = rho_valid

    return out


In [2]:


from SleepMReye.sleepmreye.sleep_mrio import SleepMRIO

mrio = SleepMRIO(
    bids_root="/Users/zach/2025_RA/matthias/bids_sleepmreye",
    exclude_subjects=["01", "07", "09", "12"],
    tr=2.1,
    task="sleep",
    load_signal=False,
)

In [8]:
from pathlib import Path
from nilearn.image import new_img_like
from mreyemove.plotting.glm import plot_img_on_surf_with_contour, custom_cmap

window_size = 3 

for state in ("W", "1", "2"):
    output_path = Path("/Users/zach/2025_RA/matthias/final images/group_level/eog_mreyemove_rho_maps") / f"{state}_correlation_map"
    
    mreyemove_group_W,_ = mrio.load_second_level_result(contrast=f"mreyemove_{state}", desc_add="clean-with-fd-clamped-original")
    eog_group_W,_ = mrio.load_second_level_result(contrast=f"eog_{state}", desc_add="eog")
    correlation_map = rolling_pearson_corr(
            mreyemove_group_W.t_stat.get_fdata(),
            eog_group_W.t_stat.get_fdata(),
            radius=window_size
        )
    correlation_img = new_img_like(mreyemove_group_W.t_stat, correlation_map, copy_header=True)
    plot_img_on_surf_with_contour(
        stat_map=correlation_img,
        views=["medial", "lateral"],
        hemispheres=["left", "right"],
        surf_mesh="fsaverage7",
        output_file=output_path,
        cmap=custom_cmap(),
        threshold=0.3,
        vmin=-1,
        vmax=1,
        bg_on_data=True,
        inflate=True,
        title=f"Spotlight Map Correlation for contrast {state}",
    )



[check_mesh_is_fsaverage] Dataset found in /Users/zach/nilearn_data/fsaverage
[check_mesh_is_fsaverage] Dataset found in /Users/zach/nilearn_data/fsaverage
[check_mesh_is_fsaverage] Dataset found in /Users/zach/nilearn_data/fsaverage


In [9]:

from pathlib import Path
from nilearn.image import new_img_like
from mreyemove.plotting.glm import plot_img_on_surf_with_contour, custom_cmap

window_size = 3 
output_dir = Path('/Users/zach/2025_RA/matthias/final images/group_level/mreyemove_state_rho_maps'
) 
desc_add = "clean-with-fd-clamped-original"
for state in ("W", "1", "2"):
    for state_2 in ("1", "2"):
        if state == state_2:
            continue
    
        print(f"Comparing {state} and {state_2}")
        mreyemove_group_1,_ = mrio.load_second_level_result(contrast=f"mreyemove_{state}", desc_add=desc_add)
        mreyemove_group_2,_ = mrio.load_second_level_result(contrast=f"mreyemove_{state_2}", desc_add=desc_add)
        correlation_map = rolling_pearson_corr(
                mreyemove_group_1.t_stat.get_fdata(),
                mreyemove_group_2.t_stat.get_fdata(),
                radius=window_size
            )
        correlation_img = new_img_like(mreyemove_group_1.t_stat, correlation_map, copy_header=True)
        plot_img_on_surf_with_contour(
            stat_map=correlation_img,
            views=["medial", "lateral"],
            hemispheres=["left", "right"],
            surf_mesh="fsaverage7",
            output_file=output_dir / f"{state}_{state_2}_correlation_map",
            cmap=custom_cmap(),
            threshold=0.3,
            vmin=-1,
            vmax=1,
            bg_on_data=True,
            inflate=True,
            title=f"Correlation for contrasts {state} vs {state_2}",
        )



Comparing W and 1
[check_mesh_is_fsaverage] Dataset found in /Users/zach/nilearn_data/fsaverage
Comparing W and 2
[check_mesh_is_fsaverage] Dataset found in /Users/zach/nilearn_data/fsaverage
Comparing 1 and W
[check_mesh_is_fsaverage] Dataset found in /Users/zach/nilearn_data/fsaverage
Comparing 1 and 2
[check_mesh_is_fsaverage] Dataset found in /Users/zach/nilearn_data/fsaverage
Comparing 2 and W
[check_mesh_is_fsaverage] Dataset found in /Users/zach/nilearn_data/fsaverage
Comparing 2 and 1
[check_mesh_is_fsaverage] Dataset found in /Users/zach/nilearn_data/fsaverage



KeyboardInterrupt



KeyboardInterrupt: 

Error in callback <function _draw_all_if_interactive at 0x309392840> (for post_execute), with arguments args (),kwargs {}:
Error in callback <function flush_figures at 0x30b2a7c40> (for post_execute), with arguments args (),kwargs {}:



KeyboardInterrupt


KeyboardInterrupt

